In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Falls das Notebook im Ordner notebooks/ liegt
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

from src.config import load_config
from src.session import get_spark_session

config = load_config().values
spark = get_spark_session()

spark.sql("""
    SELECT
        current_user() AS user,
        current_catalog() AS catalog
""").show(truncate=False)

In [1]:
from databricks.connect import DatabricksSession

spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

print("Spark-Verbindung hergestellt")

KeyboardInterrupt: 

### Check der Datensätze im Schema

In [ ]:
for catalog in ["workspace", "main"]:
    print(f"\nSchemas in {catalog}:")
    spark.sql(f"SHOW SCHEMAS IN {catalog}").show(100, truncate=False)


In [ ]:
tables_df = spark.sql("""
    SHOW TABLES IN main.dbdemos_retail_c360
""")

tables_df.show(100, truncate=False)

Die Datensätze sind als csv files abgelegt

In [ ]:
users_path = "/Volumes/main/dbdemos_retail_c360/c360/users"
orders_path = "/Volumes/main/dbdemos_retail_c360/c360/orders"
events_path = "/Volumes/main/dbdemos_retail_c360/c360/events"

users_df = spark.read.json(users_path)

orders_df = spark.read.json(orders_path)

events_df = spark.read.csv(
    events_path,
    header=True,
    inferSchema=True
)

## Users Dataframe

In [ ]:
users_df.show()

In [ ]:
from pyspark.sql import functions as F

print(f"Churn distribution:\n{users_df.groupBy('churn').count().show()}")
print(f"Distinct IDs:\n{users_df.select(
    F.countDistinct("id").alias("distinct_ids")
).first()["distinct_ids"]}")
print(f"Distinct emails:\n{users_df.select(
    F.countDistinct("email").alias("distinct_emails")
).first()["distinct_emails"]}")
print(f"Countries:\n{users_df.select(
    F.countDistinct("country").alias("distinct_countries")
).first()["distinct_countries"]}")
print(f"Countries:\n{users_df.select("country").distinct().show(truncate=False)}")

## Orders Dataframe

In [ ]:
orders_df.show()

In [ ]:
print(f"Distinct User IDs:\n{orders_df.select(
    F.countDistinct("user_id").alias("user_ids_distinct")
).first()["user_ids_distinct"]}")

print(f"Distinct IDs:\n{orders_df.select(
    F.countDistinct("id").alias("ids_distinct")
).first()["ids_distinct"]}")


In [ ]:
orders_df.filter(F.col("user_id").contains("4822f0b4")).show(truncate=False)

## Events Dataframe

In [ ]:
events_df.show()

In [ ]:
print(f"Count per Platform:\n{events_df.groupBy('platform').count().show(truncate=False)}")
print(f"Count per Action:\n{events_df.groupBy('action').count().show(truncate=False)}")
print(f"Distinct User IDs:\n{events_df.select(
    F.countDistinct("user_id").alias("user_ids_distinct")
).first()["user_ids_distinct"]}")


In [ ]:
orders_df.filter(F.col("user_id").contains("4822f0b4")).show(truncate=False)

- User ID ist duie zentrale Join Bedingugn
- Somit kann ein User mehrere Null values auseggeben haben
- Es geilt nun den Datensatz zu aggregierne udn zu Joinen

## Check for Null Values

In [ ]:
import sys

sys.path.append("../src")

from utils import get_missing_values

In [ ]:
print(f"Users Dataframe:\n{get_missing_values(users_df).show(truncate=False)}")

In [ ]:
print(f"Order Dataframe:\n{get_missing_values(orders_df).show(truncate=False)}")


In [ ]:
print(f"Events Dataframe:\n{get_missing_values(events_df).show(truncate=False)}")


In [ ]:
## Check Missing Values for complete Dataset

df_combined = users_df.join(orders_df